In [ ]:
import os
import random
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from collections import defaultdict

ROOT = Path(r"C:\D\GazProm\nogit\Digital_core_v4")
img_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp'}

# 🔹 ПОЛНЫЕ ПРАВИЛА ОБЪЕДИНЕНИЯ И УДАЛЕНИЯ (обновлено)
CLASS_RULES = {
    # ❌ УДАЛЯЕМ
    'Алевролит_песчанистый': None,
    'Песчаник_с_включениями_угля': None,
    'Известняк': None,
    'Породы_фундамента': None,

    # 📦 ГРУППА: Переслаивание_песчаника,_аргиллита_и_алевролита
    'Песчаник_с_включениями_алевролита_и_аргиллита': 'Переслаивание_песчаника,_аргиллита_и_алевролита',
    'Чередование_аргиллита,_алевролита_и_песчаника': 'Переслаивание_песчаника,_аргиллита_и_алевролита',
    'Алевролит_с_прослоями_песчаника_и_аргиллита': 'Переслаивание_песчаника,_аргиллита_и_алевролита',
    'Песчаник_с_прослоями_алевролита_и_аргиллита': 'Переслаивание_песчаника,_аргиллита_и_алевролита',
    'Аргиллит_с_прослоями_песчаника_и_алевролита': 'Переслаивание_песчаника,_аргиллита_и_алевролита',

    # 📦 ГРУППА: Уголь,_уголь_с_прослоями_аргиллита
    'Аргиллит_углистый': 'Уголь,_уголь_с_прослоями_аргиллита',
    'Уголь_с_прослоями_аргиллита': 'Уголь,_уголь_с_прослоями_аргиллита',
    'Уголь': 'Уголь,_уголь_с_прослоями_аргиллита',
    'Аргиллит_с_включениями_угля': 'Уголь,_уголь_с_прослоями_аргиллита',

    # 📦 ГРУППА: Глинисто-карбонатная_порода
    'Опока_глинистая': 'Глинисто-карбонатная_порода',
    'Глинисто-карбонатная_порода': 'Глинисто-карбонатная_порода',
    'Глина_опоковидная': 'Глинисто-карбонатная_порода',
    'Глина_аргиллитоподобная_с_прослоями_глины_опоковидной': 'Глинисто-карбонатная_порода',
    'Кремнисто-глинистая_порода': 'Глинисто-карбонатная_порода',
    'Глина_опоковидная_с_включением_глинистых_опок': 'Глинисто-карбонатная_порода',
    'Глина_аргиллитоподобная': 'Глинисто-карбонатная_порода',  # 🔹 НОВОЕ

    # 📦 ГРУППА: Алевролит
    'Алевролит_с_включениями_угля': 'Алевролит',
    'Алевролит_глинистый': 'Алевролит',
    'Алевролит_карбонатный': 'Алевролит',

    # 📦 ГРУППА: Переслаивание_аргиллита_и_алевролита
    'Аргиллит_алевритовый': 'Переслаивание_аргиллита_и_алевролита',
    'Алевролит_с_прослоями_аргиллита': 'Переслаивание_аргиллита_и_алевролита',
    'Аргиллит_с_прослоями_алевролита': 'Переслаивание_аргиллита_и_алевролита',

    # 📦 ГРУППА: Песчаник_с_прослоями_алевролита
    'Переслаивание_песчаника_и_алевролита': 'Песчаник_с_прослоями_алевролита',
    'Алевролит_с_прослоями_песчаника': 'Песчаник_с_прослоями_алевролита',

    # 📦 ГРУППА: Песчаник_с_прослоями_аргиллита
    'Аргиллит_с_прослоями_песчаника': 'Песчаник_с_прослоями_аргиллита',
    'Переслаивание_песчаника_и_аргиллита': 'Песчаник_с_прослоями_аргиллита',

    # 📦 ГРУППА: Песчаник
    'Песчаник_карбонатный': 'Песчаник',
}

# Структура: {target_group: {original_subclass: {"count": int, "all_samples": [path]}}}
hierarchy = defaultdict(lambda: defaultdict(lambda: {"count": 0, "all_samples": []}))
total_processed = 0
total_deleted = 0

print("🔍 Сканирую ТОЛЬКО модальность ДС и применяю правила...")
for dirpath, _, filenames in os.walk(ROOT):
    if 'ДС' not in Path(dirpath).parts:
        continue
        
    for f in filenames:
        if Path(f).suffix.lower() in img_exts:
            stem = Path(f).stem
            parts = stem.split('_')
            for i, part in enumerate(parts):
                if part and part[0].isdigit():
                    original_cls = '_'.join(parts[:i])
                    
                    if original_cls in CLASS_RULES:
                        target = CLASS_RULES[original_cls]
                        if target is None:
                            total_deleted += 1
                            break
                        final_group = target
                    else:
                        final_group = original_cls
                    
                    data = hierarchy[final_group][original_cls]
                    data["count"] += 1
                    data["all_samples"].append(Path(dirpath) / f)
                    total_processed += 1
                    break

def get_group_total(subclasses):
    return sum(d["count"] for d in subclasses.values())

sorted_groups = sorted(hierarchy.items(), key=lambda x: get_group_total(x[1]), reverse=True)
print(f"✅ Сканирование ДС завершено.")
print(f"   Обработано файлов: {total_processed} | Удалено: {total_deleted}")
print(f"   Осталось групп: {len(sorted_groups)}\n")

# === ВИЗУАЛИЗАЦИЯ (случайные 5 фото на подкласс) ===
rows = []
for group_name, subclasses in sorted_groups:
    total = get_group_total(subclasses)
    rows.append(("GROUP_HEADER", group_name, total, None))
    
    sorted_subs = sorted(subclasses.items(), key=lambda x: x[1]["count"], reverse=True)
    for sub_name, data in sorted_subs:
        all_samples = data["all_samples"]
        if len(all_samples) > 5:
            display_samples = random.sample(all_samples, 5)
        else:
            display_samples = all_samples
        rows.append(("SUBCLASS", sub_name, data["count"], display_samples))

fig_height = sum(3.2 if row[0] == "GROUP_HEADER" else 2.0 for row in rows) + 1
fig, axes = plt.subplots(len(rows), 6, figsize=(28, fig_height))
if len(rows) == 1:
    axes = [axes]

row_idx = 0
for row_type, name, count, samples in rows:
    if row_type == "GROUP_HEADER":
        for col in range(6):
            axes[row_idx, col].axis('off')
            if col == 0:
                axes[row_idx, col].text(0, 0.5, f"[{name}]\nВсего: {count}", 
                                       fontsize=11, fontweight='bold', va='center')
                axes[row_idx, col].add_patch(plt.Rectangle((0,0), 1, 1, facecolor='lightblue', alpha=0.15, transform=axes[row_idx, col].transAxes))
    else:
        axes[row_idx, 0].axis('off')
        axes[row_idx, 0].text(0, 0.5, f"{name}\n({count})", fontsize=9, va='center')
        
        for col in range(5):
            ax = axes[row_idx, col+1]
            ax.set_aspect('equal')
            if col < len(samples):
                try:
                    img = Image.open(samples[col])
                    ax.imshow(img)
                except Exception:
                    ax.text(0.5, 0.5, "Err", ha='center', va='center', fontsize=8)
            ax.axis('off')
            ax.set_xticks([])
            ax.set_yticks([])
            ax.margins(0)
    
    row_idx += 1

plt.tight_layout(pad=0.8, h_pad=1.0, w_pad=0.8)
plt.show()

# === ТЕКСТОВЫЙ ОТЧЕТ ===
print("\n" + "="*85)
print("ИТОГОВАЯ СТРУКТУРА ДАТАСЕТА (только модальность ДС)")
print("="*85)
for group_name, subclasses in sorted_groups:
    total = get_group_total(subclasses)
    print(f"\n📦 {group_name} — {total} изображений")
    sorted_subs = sorted(subclasses.items(), key=lambda x: x[1]["count"], reverse=True)
    for sub_name, data in sorted_subs:
        print(f"   ├─ {sub_name}: {data['count']}")

final_total = sum(get_group_total(subs) for subs in hierarchy.values())
print(f"\n📊 Общий объем датасета для обучения (ДС): {final_total} изображений")
print("="*85)
print("💡 Группировка обновлена. Запусти код ещё раз, чтобы увидеть другую случайную выборку фото.")
print("   Если всё устраивает, пиши 'Готово' → соберу PyTorch Dataset + DataLoader.")

In [ ]:
import os
from pathlib import Path
from collections import Counter, defaultdict
import pandas as pd

ROOT = Path(r"C:\D\GazProm\nogit\Digital_core_v4")
img_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp'}

# 🔹 СПИСКИ СКВАЖИН
VAL_WELLS = {"Харасавэйск_1400", "Харасавэйск_1900", "Харасавэйск_2000"}
TRAIN_WELLS = {
    "Восточно-Падинск_3-ВП", "Соболох-Неджелинск_3", "Таб-Яхинская_10360",
    "Харасавэйск_300", "Харасавэйск_600 к 6", "Харасавэйск_700",
    "Харасавэйск_900", "Харасавэйск_1100", "Харасавэйск_1800",
    "Ю-Песцовый лу_12", "Ямбург_1 N", "Ямбургск_24604"
}
ALL_WELLS = TRAIN_WELLS | VAL_WELLS

# 🔹 ПРАВИЛА ГРУППИРОВКИ
CLASS_RULES = {
    'Песчаник': 'Песчаник', 'Аргиллит': 'Аргиллит', 'Алевролит': 'Алевролит',
    'Алевролит_песчанистый': None, 'Песчаник_с_включениями_угля': None,
    'Известняк': None, 'Породы_фундамента': None,
    'Песчаник_с_включениями_алевролита_и_аргиллита': 'Переслаивание_песчаника,_аргиллита_и_алевролита',
    'Чередование_аргиллита,_алевролита_и_песчаника': 'Переслаивание_песчаника,_аргиллита_и_алевролита',
    'Алевролит_с_прослоями_песчаника_и_аргиллита': 'Переслаивание_песчаника,_аргиллита_и_алевролита',
    'Песчаник_с_прослоями_алевролита_и_аргиллита': 'Переслаивание_песчаника,_аргиллита_и_алевролита',
    'Аргиллит_с_прослоями_песчаника_и_алевролита': 'Переслаивание_песчаника,_аргиллита_и_алевролита',
    'Аргиллит_углистый': 'Уголь,_уголь_с_прослоями_аргиллита',
    'Уголь_с_прослоями_аргиллита': 'Уголь,_уголь_с_прослоями_аргиллита',
    'Уголь': 'Уголь,_уголь_с_прослоями_аргиллита',
    'Аргиллит_с_включениями_угля': 'Уголь,_уголь_с_прослоями_аргиллита',
    'Опока_глинистая': 'Глинисто-карбонатная_порода',
    'Глинисто-карбонатная_порода': 'Глинисто-карбонатная_порода',
    'Глина_опоковидная': 'Глинисто-карбонатная_порода',
    'Глина_аргиллитоподобная_с_прослоями_глины_опоковидной': 'Глинисто-карбонатная_порода',
    'Кремнисто-глинистая_порода': 'Глинисто-карбонатная_порода',
    'Глина_опоковидная_с_включением_глинистых_опок': 'Глинисто-карбонатная_порода',
    'Глина_аргиллитоподобная': 'Глинисто-карбонатная_порода',
    'Алевролит_с_включениями_угля': 'Алевролит',
    'Алевролит_глинистый': 'Алевролит',
    'Алевролит_карбонатный': 'Алевролит',
    'Аргиллит_алевритовый': 'Переслаивание_аргиллита_и_алевролита',
    'Алевролит_с_прослоями_аргиллита': 'Переслаивание_аргиллита_и_алевролита',
    'Аргиллит_с_прослоями_алевролита': 'Переслаивание_аргиллита_и_алевролита',
    'Переслаивание_песчаника_и_алевролита': 'Песчаник_с_прослоями_алевролита',
    'Алевролит_с_прослоями_песчаника': 'Песчаник_с_прослоями_алевролита',
    'Аргиллит_с_прослоями_песчаника': 'Песчаник_с_прослоями_аргиллита',
    'Переслаивание_песчаника_и_аргиллита': 'Песчаник_с_прослоями_аргиллита',
    'Песчаник_карбонатный': 'Песчаник',
}

def extract_class(filename):
    stem = Path(filename).stem
    parts = stem.split('_')
    for i, part in enumerate(parts):
        if part and part[0].isdigit(): return '_'.join(parts[:i])
    return None

def find_well(dirpath):
    for part in Path(dirpath).parts:
        if part in ALL_WELLS: return part
    return None

# Сбор статистики
train_counts = Counter()
val_counts = Counter()

print("🔍 Сканирую модальность ДС...")
for dirpath, _, filenames in os.walk(ROOT):
    if 'ДС' not in Path(dirpath).parts: continue
    
    well = find_well(dirpath)
    if not well: continue
    
    split = 'train' if well in TRAIN_WELLS else 'val'
    
    for f in filenames:
        if Path(f).suffix.lower() not in img_exts: continue
        orig_cls = extract_class(f)
        if not orig_cls: continue
        
        final_cls = CLASS_RULES.get(orig_cls, orig_cls)
        if final_cls is None: continue
        
        if split == 'train':
            train_counts[final_cls] += 1
        else:
            val_counts[final_cls] += 1

# === ВЫВОД ===
print("\n" + "="*90)
print("📊 БАЛАНС КЛАССОВ: TRAIN vs VAL (модальность ДС)")
print("="*90)

# Все классы (объединяем из обоих сплитов)
all_classes = sorted(set(train_counts.keys()) | set(val_counts.keys()))

print(f"\n{'Класс':<55} {'Train':>10} {'Val':>10} {'Доля в Train':>12}")
print("-"*90)

train_total = sum(train_counts.values())
val_total = sum(val_counts.values())

for cls in all_classes:
    t = train_counts.get(cls, 0)
    v = val_counts.get(cls, 0)
    pct = t / train_total * 100 if train_total > 0 else 0
    print(f"{cls:<55} {t:>10,} {v:>10,} {pct:>11.2f}%")

print("-"*90)
print(f"{'ИТОГО':<55} {train_total:>10,} {val_total:>10,} {'100.00%':>12}")

# 🔍 Метрики дисбаланса
print("\n" + "="*90)
print("⚖️  МЕТРИКИ ДИСБАЛАНСА")
print("="*90)

if train_counts:
    train_vals = list(train_counts.values())
    train_max, train_min = max(train_vals), min(train_vals)
    train_ratio = train_max / train_min if train_min > 0 else float('inf')
    print(f"\n🔹 TRAIN:")
    print(f"   • Максимум: {train_max:,} | Минимум: {train_min:,}")
    print(f"   • Дисбаланс (макс:мин): {train_ratio:.1f}:1")
    print(f"   • Среднее: {sum(train_vals)/len(train_vals):,.0f} | Медиана: {sorted(train_vals)[len(train_vals)//2]:,}")

if val_counts:
    val_vals = list(val_counts.values())
    val_max, val_min = max(val_vals), min(val_vals)
    val_ratio = val_max / val_min if val_min > 0 else float('inf')
    print(f"\n🔹 VAL:")
    print(f"   • Максимум: {val_max:,} | Минимум: {val_min:,}")
    print(f"   • Дисбаланс (макс:мин): {val_ratio:.1f}:1")
    print(f"   • Среднее: {sum(val_vals)/len(val_vals):,.0f} | Медиана: {sorted(val_vals)[len(val_vals)//2]:,}")

# 🔎 Самый маленький класс в Train
if train_counts:
    smallest_train = min(train_counts, key=train_counts.get)
    print(f"\n⚠️  Самый маленький класс в Train: '{smallest_train}' ({train_counts[smallest_train]:,} фото)")
    print(f"   → Рекомендуется: WeightedRandomSampler + class_weights в Loss")

print("\n" + "="*90)

In [ ]:
import os
import shutil
from pathlib import Path
from collections import defaultdict

# 🔹 НАСТРОЙКИ
SOURCE_ROOT = Path(r"C:\D\GazProm\nogit\Digital_core_v4")
TARGET_ROOT = Path(r"C:\D\GazProm\nogit\Digital_core_tmv2")

img_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp'}

# 🔹 СПИСКИ СКВАЖИН ПО СПЛИТАМ
VAL_WELLS = {"Харасавэйск_1400", "Харасавэйск_1900", "Харасавэйск_2000"}
TRAIN_WELLS = {
    "Восточно-Падинск_3-ВП", "Соболох-Неджелинск_3", "Таб-Яхинская_10360",
    "Харасавэйск_300", "Харасавэйск_600 к 6", "Харасавэйск_700",
    "Харасавэйск_900", "Харасавэйск_1100", "Харасавэйск_1800",
    "Ю-Песцовый лу_12", "Ямбург_1 N", "Ямбургск_24604"
}
TEST_WELLS = {"Харасавэйск_1700"}
ALL_WELLS = TRAIN_WELLS | VAL_WELLS | TEST_WELLS

# 🔹 ПРАВИЛА ГРУППИРОВКИ
CLASS_RULES = {
    'Песчаник': 'Песчаник', 'Аргиллит': 'Аргиллит', 'Алевролит': 'Алевролит',
    'Алевролит_песчанистый': None, 'Песчаник_с_включениями_угля': None,
    'Известняк': None, 'Породы_фундамента': None,
    'Песчаник_с_включениями_алевролита_и_аргиллита': 'Переслаивание_песчаника,_аргиллита_и_алевролита',
    'Чередование_аргиллита,_алевролита_и_песчаника': 'Переслаивание_песчаника,_аргиллита_и_алевролита',
    'Алевролит_с_прослоями_песчаника_и_аргиллита': 'Переслаивание_песчаника,_аргиллита_и_алевролита',
    'Песчаник_с_прослоями_алевролита_и_аргиллита': 'Переслаивание_песчаника,_аргиллита_и_алевролита',
    'Аргиллит_с_прослоями_песчаника_и_алевролита': 'Переслаивание_песчаника,_аргиллита_и_алевролита',
    'Аргиллит_углистый': 'Уголь,_уголь_с_прослоями_аргиллита',
    'Уголь_с_прослоями_аргиллита': 'Уголь,_уголь_с_прослоями_аргиллита',
    'Уголь': 'Уголь,_уголь_с_прослоями_аргиллита',
    'Аргиллит_с_включениями_угля': 'Уголь,_уголь_с_прослоями_аргиллита',
    'Опока_глинистая': 'Глинисто-карбонатная_порода',
    'Глинисто-карбонатная_порода': 'Глинисто-карбонатная_порода',
    'Глина_опоковидная': 'Глинисто-карбонатная_порода',
    'Глина_аргиллитоподобная_с_прослоями_глины_опоковидной': 'Глинисто-карбонатная_порода',
    'Кремнисто-глинистая_порода': 'Глинисто-карбонатная_порода',
    'Глина_опоковидная_с_включением_глинистых_опок': 'Глинисто-карбонатная_порода',
    'Глина_аргиллитоподобная': 'Глинисто-карбонатная_порода',
    'Алевролит_с_включениями_угля': 'Алевролит',
    'Алевролит_глинистый': 'Алевролит',
    'Алевролит_карбонатный': 'Алевролит',
    'Аргиллит_алевритовый': 'Переслаивание_аргиллита_и_алевролита',
    'Алевролит_с_прослоями_аргиллита': 'Переслаивание_аргиллита_и_алевролита',
    'Аргиллит_с_прослоями_алевролита': 'Переслаивание_аргиллита_и_алевролита',
    'Переслаивание_песчаника_и_алевролита': 'Песчаник_с_прослоями_алевролита',
    'Алевролит_с_прослоями_песчаника': 'Песчаник_с_прослоями_алевролита',
    'Аргиллит_с_прослоями_песчаника': 'Песчаник_с_прослоями_аргиллита',
    'Переслаивание_песчаника_и_аргиллита': 'Песчаник_с_прослоями_аргиллита',
    'Песчаник_карбонатный': 'Песчаник',
}

def extract_class(filename):
    """Извлекает название класса из имени файла (до первой цифры)"""
    stem = Path(filename).stem
    parts = stem.split('_')
    for i, part in enumerate(parts):
        if part and part[0].isdigit():
            return '_'.join(parts[:i])
    return None

def find_well(dirpath):
    """Ищет название скважины в пути"""
    for part in Path(dirpath).parts:
        if part in ALL_WELLS:
            return part
    return None

# ==========================================
# ОСНОВНОЙ ПРОЦЕСС
# ==========================================
stats = defaultdict(int)
total_copied = 0
total_deleted = 0

print("🔍 Сканирую исходную папку и раскладываю файлы...")
for dirpath, _, filenames in os.walk(SOURCE_ROOT):
    # 1. Определяем модальность
    modality = None
    for part in Path(dirpath).parts:
        if part == "ДС": modality = "ДС"; break
        if part == "УФ": modality = "УФ"; break
    if not modality: continue

    # 2. Определяем скважину и сплит
    well = find_well(dirpath)
    if not well: continue # Пропускаем, если скважина не из списка
    
    if well in TRAIN_WELLS:
        split = "train"
    elif well in VAL_WELLS:
        split = "val"
    elif well in TEST_WELLS:
        split = "test"
    else:
        continue # Скважины, не попавшие ни в один список (например, старые тесты)

    # 3. Копируем файлы
    for f in filenames:
        if Path(f).suffix.lower() not in img_exts: continue
        
        src_file = Path(dirpath) / f
        orig_cls = extract_class(f)
        
        if not orig_cls: continue
            
        # Применяем правила группировки
        target_cls = CLASS_RULES.get(orig_cls, orig_cls)
        
        if target_cls is None:
            total_deleted += 1
            continue # Удаляемые классы пропускаем

        # Формируем целевой путь: .../Digital_core_tmv2/ДС/train/Песчаник/
        dest_dir = TARGET_ROOT / modality / split / target_cls
        dest_dir.mkdir(parents=True, exist_ok=True)
        
        shutil.copy2(src_file, dest_dir / f)
        
        stats[f"{modality}_{split}_{target_cls}"] += 1
        total_copied += 1
        
        if total_copied % 2000 == 0:
            print(f"  ⏳ Скопировано: {total_copied:,} файлов...", end='\r')

print("\n✅ Готово!")
print(f"📂 Цель: {TARGET_ROOT}")

# 📊 Статистика по итогу
print("\n📊 ИТОГОВАЯ СТАТИСТИКА:")
for mod in ["ДС", "УФ"]:
    for s in ["train", "val", "test"]:
        count = sum(v for k, v in stats.items() if k.startswith(f"{mod}_{s}"))
        print(f"   {mod} / {s}: {count:,} файлов")

print(f"\n❌ Всего удалено (по правилам): {total_deleted}")

In [ ]:
import os
from pathlib import Path
from PIL import Image
from collections import defaultdict

# 🔹 ПУТЬ К НОВОМУ ДАТАСЕТУ
DATASET_ROOT = Path(r"C:\D\GazProm\nogit\Digital_core_tmv2")

# 🔹 УТВЕРЖДЁННЫЕ КЛАССЫ (9 штук)
VALID_CLASSES = {
    'Песчаник',
    'Аргиллит', 
    'Алевролит',
    'Переслаивание_песчаника,_аргиллита_и_алевролита',
    'Переслаивание_аргиллита_и_алевролита',
    'Песчаник_с_прослоями_алевролита',
    'Песчаник_с_прослоями_аргиллита',
    'Глинисто-карбонатная_порода',
    'Уголь,_уголь_с_прослоями_аргиллита'
}

img_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp'}

print("🔍 Запускаю быструю валидацию датасета...")
print(f"📂 Проверяю: {DATASET_ROOT}\n")

stats = defaultdict(lambda: {"total": 0, "ok": 0, "errors": []})
total_files = 0
total_errors = 0

for modality in ["ДС", "УФ"]:
    mod_path = DATASET_ROOT / modality
    if not mod_path.exists():
        print(f"⚠️  Папка {modality} не найдена, пропускаем.")
        continue
        
    for split in ["train", "val", "test"]:
        split_path = mod_path / split
        if not split_path.exists():
            continue
            
        for class_folder in split_path.iterdir():
            if not class_folder.is_dir():
                continue
                
            class_name = class_folder.name
            
            # Проверка: валидный ли класс?
            if class_name not in VALID_CLASSES:
                stats[f"{modality}/{split}"]["errors"].append(f"❌ Неизвестный класс: {class_name}")
                total_errors += 1
                continue
            
            # Проверка файлов в классе
            for file_path in class_folder.iterdir():
                if file_path.suffix.lower() not in img_exts:
                    continue  # Пропускаем не-изображения (например, .DS_Store)
                
                stats[f"{modality}/{split}"]["total"] += 1
                total_files += 1
                
                try:
                    img = Image.open(file_path)
                    img.load()  # Принудительно загружаем данные
                    stats[f"{modality}/{split}"]["ok"] += 1
                except Exception as e:
                    error_msg = f"🔴 {file_path.name}: {type(e).__name__}"
                    stats[f"{modality}/{split}"]["errors"].append(error_msg)
                    total_errors += 1

# === ОТЧЁТ ===
print("\n" + "="*80)
print("📊 РЕЗУЛЬТАТЫ ВАЛИДАЦИИ")
print("="*80)

for key in sorted(stats.keys()):
    data = stats[key]
    total, ok, errors = data["total"], data["ok"], data["errors"]
    pct = ok / total * 100 if total > 0 else 100
    
    status = "✅" if len(errors) == 0 else "⚠️"
    print(f"\n{status} {key}:")
    print(f"   Всего файлов: {total:,} | Открылось: {ok:,} ({pct:.2f}%)")
    
    if errors:
        print(f"   ❌ Ошибки ({len(errors)}):")
        for err in errors[:10]:  # Показываем первые 10
            print(f"      • {err}")
        if len(errors) > 10:
            print(f"      ... и ещё {len(errors) - 10} ошибок")

print("\n" + "="*80)
print(f"📈 ОБЩИЕ ИТОГИ:")
print(f"   Проверено файлов: {total_files:,}")
print(f"   Ошибок: {total_errors}")
if total_errors == 0:
    print("   ✅ ✅ ✅ ВСЕ ФАЙЛЫ ВАЛИДНЫ! МОЖНО ОБУЧАТЬ! ✅ ✅ ✅")
else:
    print(f"   ⚠️  Процент успеха: {(total_files - total_errors) / total_files * 100:.4f}%")
print("="*80)

In [ ]:
import json
from pathlib import Path

# 🔹 УТВЕРЖДЁННЫЕ КЛАССЫ (в том порядке, в котором хочешь индексы)
# Порядок важен: первый класс получит индекс 0, второй — 1, и т.д.
CLASSES_ORDER = [
    'Песчаник',
    'Аргиллит', 
    'Алевролит',
    'Переслаивание_песчаника,_аргиллита_и_алевролита',
    'Переслаивание_аргиллита_и_алевролита',
    'Песчаник_с_прослоями_алевролита',
    'Песчаник_с_прослоями_аргиллита',
    'Глинисто-карбонатная_порода',
    'Уголь,_уголь_с_прослоями_аргиллита'
]

# Создаём маппинг: класс → индекс
class_to_idx = {cls: idx for idx, cls in enumerate(CLASSES_ORDER)}

# 🔹 СОХРАНЯЕМ В ФАЙЛ
output_path = Path(r"C:\D\GazProm\nogit\Digital_core_tmv2\label_encoder.json")
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(class_to_idx, f, ensure_ascii=False, indent=2)

print(f"✅ label_encoder.json сохранён: {output_path}")
print("\n📋 Содержимое файла:")
for cls, idx in class_to_idx.items():
    print(f"   {idx}: {cls}")